# AlphaTrain Iteration 1b: Mixed Training (Colab)

Train the ResNet on **554K states**: 50% expert heuristic + 50% sharpened self-play (T=0.1).
Policy learns sharp MCTS visit distributions. Value learns TD returns (MSE).

**Requires A100 runtime** (~20 GB VRAM for data + model).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (81 KB)
2. `mixed_iter1.pt.gz` — training data (178 MB compressed, 17 GB uncompressed)
3. `alphatrain_td_best.pt` — base model to fine-tune

**Build locally:**
```bash
# Tarball (if not already built)
tar czf colorlines_selfplay_train.tar.gz \
    --exclude='__pycache__' --exclude='*.nbc' --exclude='*.nbi' \
    --exclude='data' --exclude='.venv' --exclude='.git' \
    --exclude='*.tar.gz' --exclude='selfplay_data' \
    -C /Users/andreis/local/source/colorlines98 \
    alphatrain game

# Compress tensor (17 GB -> 178 MB)
gzip -c alphatrain/data/mixed_iter1.pt > mixed_iter1.pt.gz
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)

# Decompress training data (178 MB gz -> 17 GB pt)
data_dst = '/content/alphatrain/data/mixed_iter1.pt'
if not os.path.exists(data_dst):
    print('Decompressing mixed_iter1.pt.gz...')
    !gunzip -c {DRIVE}/mixed_iter1.pt.gz > {data_dst}
print(f'Data: {os.path.getsize(data_dst)/1e9:.1f} GB')

# Copy model
model_dst = '/content/alphatrain/data/alphatrain_td_best.pt'
if not os.path.exists(model_dst):
    shutil.copy(f'{DRIVE}/alphatrain_td_best.pt', model_dst)
print(f'Model: {os.path.getsize(model_dst)/1e6:.0f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 20e9:
        print('WARNING: T4 may not have enough VRAM. Use A100 runtime.')

!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Iteration 1b: mixed training (50% expert + 50% sharpened self-play)
# Policy: KL-div with sharp targets (entropy 0.28, was 3.82 at T=1.0)
# Value: MSE on scalar TD returns (sigmoid * 500)
# LR 1e-4 (gentle, prevents catastrophic forgetting)
%cd /content
!python -m alphatrain.train \
    --tensor-file alphatrain/data/mixed_iter1.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 1.0 \
    --resume alphatrain/data/alphatrain_td_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/selfplay_iter1_best.pt \
    --save-dir /content/checkpoints/selfplay_iter1

In [ ]:
# Evaluate: standalone policy score (baseline was 314)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/selfplay_iter1/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/selfplay_iter1/{f}'
    dst = f'{DRIVE}/selfplay_iter1_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst}')